In [48]:
import openai
import instructor
import pandas as pd

from pydantic import BaseModel, Field
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Filter,
    FieldCondition,
    MatchValue,
    VectorParams,
    Distance,
    SparseVectorParams,
    Modifier,
    PayloadSchemaType,
    Document,
    PointStruct,
    Prefetch,
    FusionQuery,
    RrfQuery,
    Rrf
)

### Qdrant collection for Hybrid Search

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [8]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-01-hybrid-search",
    vectors_config={
        "text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE)
    },
    sparse_vectors_config={
        "bm25": SparseVectorParams(modifier=Modifier.IDF)
    }
)

True

In [10]:
qdrant_client.create_payload_index(
    collection_name="Amazon-items-collection-01-hybrid-search",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

### Embedding Functions

In [38]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [12]:
def get_embedding_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

### Read the sampled dataset with Amazon inventory data

In [14]:
data = pd.read_json("../../data/meta_Books_2022_2023_with_category_ratings_10_sample_1000.jsonl", lines=True)

### Preprocess Title and Features

In [16]:
def process_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

def extract_first_large_image(row):
    return row['images'][0].get('large', '') if row['images'] else ''

In [17]:
data['description'] = data.apply(process_description, axis=1)
data['image'] = data.apply(extract_first_large_image, axis=1)

In [18]:
data_to_embed = data[
    ["title", "description", "categories_middle", "image", "rating_number", "average_rating", "price", "parent_asin"]
].to_dict(orient="records")

In [20]:
len(data_to_embed)

1000

### Embed Data

#### Amazon data

In [21]:
text_to_embed = [item["description"] for item in data_to_embed]

In [24]:
dense_embeddings = get_embedding_batch(text_to_embed)

Processed 100 of 1000
Processed 200 of 1000
Processed 300 of 1000
Processed 400 of 1000
Processed 500 of 1000
Processed 600 of 1000
Processed 700 of 1000
Processed 800 of 1000
Processed 900 of 1000
Processed 1000 of 1000


In [29]:
pointstructs = []
idx = 1
for embedding, data in zip(dense_embeddings, data_to_embed):
    pointstructs.append(
        PointStruct(
            id=idx,
            vector={
                "text-embedding-3-small": embedding,
                "bm25": Document(
                    text=data["description"],
                    model="qdrant/bm25",
                )
            },
            payload=data
        )
    )
    idx += 1

In [31]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01-hybrid-search",
    points=pointstructs[0:500],
    wait=True
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [33]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01-hybrid-search",
    points=pointstructs[500:],
    wait=True
)

UpdateResult(operation_id=4, status=<UpdateStatus.COMPLETED: 'completed'>)

### Hybrid Retrieval

In [44]:
def retrieve_data(query, qdrant_client, k=5):
   
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25",
                ),
                using="bm25",
                limit=20
            )
        ],
        query=FusionQuery(fusion='rrf'),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [45]:
result = retrieve_data("What are the most popular self improvement books?", qdrant_client, k=20)

In [46]:
result

{'retrieved_context_ids': ['B0B2TW64X7',
  '1778002935',
  'B0BXN7F5Z6',
  'B0BM3VQD48',
  '1953156614',
  '1401971903',
  '8409504545',
  '0744039894',
  '1942121628',
  '0593201914',
  '1915036747',
  'B0B95WSZV9',
  '1958714712',
  '1961970023',
  '195780761X',
  'B0BN1STM2L',
  '057831679X',
  '1739276302',
  '1737911728',
  '161599601X'],
 'retrieved_context': ['Becoming the Change: Five Essential Elements to Being Your Best Self Becoming The Change provides the reader with the tools and resources necessary to manage the one thing we can truly control—ourselves. Wolfe discusses change much like the metamorphosis of a butterﬂy, whereas we experience our own type of metamorphosis as it relates to our own social and emotional well-being. Five innate qualities are instilled deep within our core, our moral compass, known as Social Emotional Learning (SEL). At its center is Self-Awareness which directly impacts the four directional pathways of Self-Management, Social Awareness, Relation

### Hybrid Search with Weighted RRF

In [52]:
def retrieve_data(query, qdrant_client, k=5):
   
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25",
                ),
                using="bm25",
                limit=20
            )
        ],
        query=RrfQuery(rrf=Rrf(weights=[3,1])),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [53]:
result = retrieve_data("What are the most popular self improvement books?", qdrant_client, k=20)

In [54]:
result

{'retrieved_context_ids': ['B0B2TW64X7',
  'B0BXN7F5Z6',
  '1401971903',
  '1778002935',
  '1953156614',
  '1942121628',
  'B0BM3VQD48',
  '1915036747',
  '1958714712',
  '0593201914',
  '195780761X',
  '057831679X',
  '1737911728',
  '161599601X',
  '8409504545',
  '1250878330',
  '1728234891',
  '0578376490',
  '0744039894',
  '1803614625'],
 'retrieved_context': ['Becoming the Change: Five Essential Elements to Being Your Best Self Becoming The Change provides the reader with the tools and resources necessary to manage the one thing we can truly control—ourselves. Wolfe discusses change much like the metamorphosis of a butterﬂy, whereas we experience our own type of metamorphosis as it relates to our own social and emotional well-being. Five innate qualities are instilled deep within our core, our moral compass, known as Social Emotional Learning (SEL). At its center is Self-Awareness which directly impacts the four directional pathways of Self-Management, Social Awareness, Relation